# **Understanding & Cleaning the Datasets**

# All Imports Here

In [354]:
import os

gl_path = "D:\\GitHub\\Falconstack_SaaS_Growth_Retention"

os.chdir(gl_path)
os.getcwd()

'D:\\GitHub\\Falconstack_SaaS_Growth_Retention'

In [355]:
from Reusable_Python_Data.filename_to_df import Filenames_To_Dataframes
import pandas as pd
import numpy as np


# Insert Datasets

In [356]:
fn_df = Filenames_To_Dataframes(f"{gl_path}\\data\\raw")
dfs, fns = fn_df.get_dataframes()

In [357]:
fns

['accounts.csv',
 'churn_events.csv',
 'feature_usage.csv',
 'subscriptions.csv',
 'support_tickets.csv']

In [358]:
accounts_df = dfs[0]
churn_events_df = dfs[1]
feature_usage_df = dfs[2]
subscriptions_df = dfs[3]
support_tickets_df = dfs[4]

# Checking Individual Dataset

## *Accounts*

In [359]:
accounts_df.head()

,account_id,account_name,industry,country,signup_date,referral_source,plan_tier,seats,is_trial,churn_flag
0,A-2e4581,Company_0,EdTech,US,16-10-2024,partner,Basic,9,False,False
1,A-43a9e3,Company_1,FinTech,IN,17-08-2023,other,Basic,18,False,True
2,A-0a282f,Company_2,DevTools,US,27-08-2024,organic,Basic,1,False,False
3,A-1f0ac7,Company_3,HealthTech,UK,27-08-2023,other,Basic,24,True,False
4,A-ce550d,Company_4,HealthTech,US,27-10-2024,event,Enterprise,35,False,True


In [360]:
accounts_df.shape

(550, 10)

In [361]:
print(f"Null counts: {accounts_df.isnull().sum()}\n")
print(f"Duplicate counts: {accounts_df.duplicated().sum()}")

Null counts: account_id         0
account_name       0
industry           8
country            6
signup_date        0
referral_source    0
plan_tier          0
seats              0
is_trial           0
churn_flag         0
dtype: int64

Duplicate counts: 8


In [362]:
accounts_df[accounts_df['industry'].isnull()]

,account_id,account_name,industry,country,signup_date,referral_source,plan_tier,seats,is_trial,churn_flag
502,A-000003,Company_374,NaN,US,07-11-2024,other,Enterprise,39,False,True
504,A-9289f6,Company_104,NaN,US,31-05-2023,event,Enterprise,1,False,True
508,A-84ebe4,Company_68,NaN,US,12-03-2023,ads,Pro,18,True,True
512,A-bbe56f,Company_406,NaN,US,16-02-2023,ads,Enterprise,21,False,False
515,A-00000b,Company_388,NaN,FR,31-12-2024,partner,Enterprise,3,False,False
526,A-d26ab4,Company_497,NaN,UK,07-11-2024,organic,Basic,9,False,True
532,A-cc1d8d,Company_440,NaN,US,08-12-2024,other,Pro,5,False,False
549,A-000023,Company_22,NaN,US,20-08-2024,ads,Enterprise,3,False,True


In [363]:
accounts_df[accounts_df['country'].isnull()]

,account_id,account_name,industry,country,signup_date,referral_source,plan_tier,seats,is_trial,churn_flag
507,A-000006,Company_124,DevTools,NaN,19-06-2023,other,Enterprise,1,False,True
517,A-396e5f,Company_30,Cybersecurity,NaN,02-10-2024,other,Enterprise,2,False,False
523,A-000011,Company_356,Cybersecurity,NaN,28-05-2023,other,Pro,8,False,False
533,A-000017,Company_173,FinTech,NaN,02-10-2024,organic,Basic,3,False,True
534,A-000018,Company_2,DevTools,NaN,27-08-2024,organic,Basic,1,False,False
542,A-00001e,Company_485,EdTech,NaN,31-07-2024,partner,Enterprise,53,False,False


In [364]:
accounts_df[accounts_df['account_name'] == 'Company_124']

,account_id,account_name,industry,country,signup_date,referral_source,plan_tier,seats,is_trial,churn_flag
124,A-5a184f,Company_124,DevTools,DE,19-06-2023,other,Enterprise,1,False,True
507,A-000006,Company_124,DevTools,NaN,19-06-2023,other,Enterprise,1,False,True


In [365]:
# insight: using account name to get the corresponding industry or country name

# null countries
accounts_df['country'] = accounts_df['country'].fillna(
    accounts_df.groupby('account_name')['country'].transform('first')
)

# null industries
accounts_df['industry'] = accounts_df['industry'].fillna(
    accounts_df.groupby('account_name')['industry'].transform('first')
)

In [366]:
accounts_df.isnull().sum()

account_id         0
account_name       0
industry           0
country            0
signup_date        0
referral_source    0
plan_tier          0
seats              0
is_trial           0
churn_flag         0
dtype: int64

In [367]:
print(f"Duplicate counts: {accounts_df.duplicated().sum()}")

Duplicate counts: 13


In [368]:
accounts_df = accounts_df.drop_duplicates()
print(f"Duplicate counts: {accounts_df.duplicated().sum()}")

Duplicate counts: 0


In [369]:
accounts_df.info()

<class 'pandas.DataFrame'>
Index: 537 entries, 0 to 549
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   account_id       537 non-null    str  
 1   account_name     537 non-null    str  
 2   industry         537 non-null    str  
 3   country          537 non-null    str  
 4   signup_date      537 non-null    str  
 5   referral_source  537 non-null    str  
 6   plan_tier        537 non-null    str  
 7   seats            537 non-null    int64
 8   is_trial         537 non-null    bool 
 9   churn_flag       537 non-null    bool 
dtypes: bool(2), int64(1), str(7)
memory usage: 38.8 KB


In [370]:
accounts_df.describe()


,seats
count,537.000000
mean,20.445065
std,20.750860
min,1.000000
25%,5.000000
50%,15.000000
75%,28.000000
max,163.000000


## *Churn Events*

In [371]:
churn_events_df.head()

,churn_event_id,account_id,churn_date,reason_code,refund_amount_usd,preceding_upgrade_flag,preceding_downgrade_flag,is_reactivation,feedback_text
0,C-816288,A-c37cab,27-10-2024,pricing,4.03,False,False,False,switched to competitor
1,C-5a81e7,A-37f969,25-06-2024,support,96.45,True,False,False,NaN
2,C-a174be,A-b07346,12-11-2024,budget,0.00,False,False,False,missing features
3,C-accb39,A-1e50e0,01-11-2023,budget,54.94,False,False,False,switched to competitor
4,C-92f889,A-956988,30-12-2024,unknown,0.00,False,True,True,too expensive


In [372]:
churn_events_df.shape

(660, 9)

In [373]:
print(f'Null counts: {churn_events_df.isnull().sum()}\n')
print(f'Duplicate counts: {churn_events_df.duplicated().sum()}\n')

Null counts: churn_event_id                0
account_id                    0
churn_date                    0
reason_code                  14
refund_amount_usd             0
preceding_upgrade_flag        0
preceding_downgrade_flag      0
is_reactivation               0
feedback_text               171
dtype: int64

Duplicate counts: 14



In [374]:
# ignoring feedback text as it might not be helpful in the future
# checking for reason code only
churn_events_df[churn_events_df['reason_code'].isnull()]

,churn_event_id,account_id,churn_date,reason_code,refund_amount_usd,preceding_upgrade_flag,preceding_downgrade_flag,is_reactivation,feedback_text
616,C-00000a,A-78f02b,02-12-2024,NaN,0.00,False,False,False,missing features
618,C-00000b,A-cdf020,08-12-2023,NaN,105.26,False,False,False,too expensive
622,C-00000f,A-6f50ae,06-11-2023,NaN,0.00,False,False,False,missing features
623,C-000010,A-dbc825,03-02-2024,NaN,0.00,False,False,False,NaN
627,C-000014,A-18793f,30-12-2024,NaN,25.41,True,False,False,NaN
629,C-000016,A-05f0e5,25-11-2024,NaN,72.19,True,False,False,NaN
633,C-57a768,A-702032,18-04-2024,NaN,10.50,False,False,False,switched to competitor
634,C-000019,A-3b5cd1,21-01-2024,NaN,0.00,True,False,False,missing features
637,C-00001b,A-fd7ad3,23-11-2024,NaN,0.00,False,False,False,too expensive
641,C-00001f,A-44dc83,29-11-2024,NaN,98.48,False,True,False,missing features


In [375]:
churn_events_df['reason_code'].unique()

<StringArray>
['pricing', 'support', 'budget', 'unknown', 'features', 'competitor', nan]
Length: 7, dtype: str

In [376]:
churn_events_df[churn_events_df.duplicated(subset=["account_id", "churn_date"], keep=False)].sort_values(["account_id", "churn_date"])

,churn_event_id,account_id,churn_date,reason_code,refund_amount_usd,preceding_upgrade_flag,preceding_downgrade_flag,is_reactivation,feedback_text
356,C-89198d,A-038089,21-10-2024,competitor,0.00,False,False,False,NaN
655,C-000027,A-038089,21-10-2024,competitor,0.00,False,False,False,NaN
132,C-70de89,A-039727,22-12-2024,pricing,46.49,False,False,False,switched to competitor
636,C-70de89,A-039727,22-12-2024,pricing,46.49,False,False,False,NaN
441,C-8f8593,A-05d3b3,09-09-2024,unknown,0.00,False,False,False,switched to competitor
...,...,...,...,...,...,...,...,...,...
640,C-00001e,A-dfbd31,01-01-2024,features,0.00,True,False,False,NaN
570,C-2de339,A-e83938,29-12-2024,budget,13.50,False,False,False,NaN
643,C-000021,A-e83938,29-12-2024,budget,13.50,False,False,False,NaN
515,C-2299d7,A-fd7ad3,23-11-2024,support,0.00,False,False,False,too expensive


In [377]:
# insight: account id and churn date are same meaning there are duplicate records 
# keep the duplicate record which has the reason code and throw the null one
churn_events_df = churn_events_df[
    ~(
        churn_events_df.duplicated(subset=["account_id", "churn_date"], keep=False)
        & churn_events_df['reason_code'].isna()
    )
]

churn_events_df.isnull().sum()

churn_event_id                0
account_id                    0
churn_date                    0
reason_code                   0
refund_amount_usd             0
preceding_upgrade_flag        0
preceding_downgrade_flag      0
is_reactivation               0
feedback_text               166
dtype: int64

In [378]:
churn_events_df = churn_events_df.drop_duplicates()

In [379]:
print(f'Duplicate counts: {churn_events_df.duplicated().sum()}\n')

Duplicate counts: 0



In [380]:
churn_events_df.info()

<class 'pandas.DataFrame'>
Index: 632 entries, 0 to 658
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   churn_event_id            632 non-null    str    
 1   account_id                632 non-null    str    
 2   churn_date                632 non-null    str    
 3   reason_code               632 non-null    str    
 4   refund_amount_usd         632 non-null    float64
 5   preceding_upgrade_flag    632 non-null    bool   
 6   preceding_downgrade_flag  632 non-null    bool   
 7   is_reactivation           632 non-null    bool   
 8   feedback_text             469 non-null    str    
dtypes: bool(3), float64(1), str(5)
memory usage: 36.4 KB


In [381]:
churn_events_df.describe()

,refund_amount_usd
count,632.000000
mean,14.931946
std,41.035709
min,0.000000
25%,0.000000
50%,0.000000
75%,0.000000
max,392.920000


## *Feature Usage*

In [382]:
feature_usage_df.head()

,usage_id,subscription_id,usage_date,feature_name,usage_count,usage_duration_secs,error_count,is_beta_feature
0,U-1c6c24,S-0fcf7d,27-07-2023,feature_20,9,5004,0,False
1,U-f07cb8,S-c25263,07-08-2023,feature_5,9,369,0,False
2,U-096807,S-f29e7f,07-12-2023,feature_3,9,1458,0,False
3,U-6b1580,S-be655e,28-07-2024,feature_40,5,2085,0,False
4,U-720a29,S-f9b1d0,02-12-2024,feature_12,12,900,0,False


In [383]:
feature_usage_df.shape

(27500, 8)

In [384]:
print(f"Null counts: {feature_usage_df.isnull().sum()}\n")
print(f"Duplicate counts: {feature_usage_df.duplicated().sum()}\n")

Null counts: usage_id                 0
subscription_id          0
usage_date               0
feature_name           292
usage_count              0
usage_duration_secs      0
error_count              0
is_beta_feature          0
dtype: int64

Duplicate counts: 555



In [385]:
feature_usage_df[feature_usage_df['feature_name'].isnull()]

,usage_id,subscription_id,usage_date,feature_name,usage_count,usage_duration_secs,error_count,is_beta_feature
25002,U-000003,S-c7f25a,13-12-2024,NaN,4,1096,0,False
25007,U-000008,S-cb38be,26-01-2023,NaN,7,-1,0,False
25014,U-00000d,S-1c06f3,22-09-2023,NaN,8,168,2,False
25018,U-ccdd45,S-02be35,15-10-2024,NaN,11,1694,0,False
25025,U-4e954a,S-2fe253,15-02-2024,NaN,9,4446,0,False
...,...,...,...,...,...,...,...,...
27424,U-0006a3,S-f5f5b3,19-02-2024,NaN,11,286,0,False
27448,U-0006b7,S-e7f571,25-12-2024,NaN,13,-1,0,False
27450,U-0006b8,S-7dab19,01-03-2023,NaN,11,5390,2,False
27456,U-0006bc,S-d48577,15-10-2024,NaN,10,430,0,False


In [386]:
feature_usage_df[feature_usage_df.duplicated(subset=['subscription_id', 'usage_date'], keep=False)].sort_values(['subscription_id', 'usage_date'])

,usage_id,subscription_id,usage_date,feature_name,usage_count,usage_duration_secs,error_count,is_beta_feature
23589,U-6aca17,S-001561,25-06-2024,feature_39,8,3024,0,False
25098,U-00004e,S-001561,25-06-2024,feature_39,8,3024,0,False
5749,U-ea0de3,S-0027d3,06-05-2023,feature_23,8,2568,0,False
25890,U-000270,S-0027d3,06-05-2023,feature_23,8,2568,0,False
7290,U-e982b9,S-003647,02-06-2023,feature_38,7,1484,0,False
...,...,...,...,...,...,...,...,...
25179,U-a4af4c,S-ff582b,24-10-2024,feature_3,11,6050,3,False
397,U-a3252f,S-ff78bf,06-10-2024,feature_15,9,1431,0,False
25236,U-0000ac,S-ff78bf,06-10-2024,NaN,9,1431,0,False
1297,U-516600,S-ffe125,12-10-2024,feature_35,13,6318,0,False


In [387]:
# insight: similart problem like in churn data
feature_usage_df = feature_usage_df[
    ~(
        feature_usage_df.duplicated(subset=['subscription_id', 'usage_date'], keep=False)
        & feature_usage_df['feature_name'].isna()
    )
]

feature_usage_df.isnull().sum()

usage_id               0
subscription_id        0
usage_date             0
feature_name           0
usage_count            0
usage_duration_secs    0
error_count            0
is_beta_feature        0
dtype: int64

In [388]:
feature_usage_df = feature_usage_df.drop_duplicates()
print(f"Duplicate counts: {feature_usage_df.duplicated().sum()}\n")

Duplicate counts: 0



In [389]:
feature_usage_df.info()

<class 'pandas.DataFrame'>
Index: 26653 entries, 0 to 27498
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   usage_id             26653 non-null  str  
 1   subscription_id      26653 non-null  str  
 2   usage_date           26653 non-null  str  
 3   feature_name         26653 non-null  str  
 4   usage_count          26653 non-null  int64
 5   usage_duration_secs  26653 non-null  int64
 6   error_count          26653 non-null  int64
 7   is_beta_feature      26653 non-null  bool 
dtypes: bool(1), int64(3), str(4)
memory usage: 1.7 MB


In [390]:
feature_usage_df.describe()

,usage_count,usage_duration_secs,error_count
count,26653.000000,26653.000000,26653.000000
mean,9.954302,3019.451581,0.563576
std,3.246388,2064.483269,1.011667
min,-1.000000,-1.000000,0.000000
25%,8.000000,1320.000000,0.000000
50%,10.000000,2737.000000,0.000000
75%,12.000000,4390.000000,1.000000
max,26.000000,12696.000000,8.000000


## *Subscriptions*

In [391]:
subscriptions_df.head()

,subscription_id,account_id,start_date,end_date,plan_tier,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag
0,S-8cec59,A-3c1a3f,23-12-2023,12-04-2024,Enterprise,14,2786.0,33432.0,False,False,False,True,monthly,True
1,S-0f6f44,A-9b9fe9,11-06-2024,NaN,Pro,17,833.0,9996.0,False,False,False,False,monthly,True
2,S-51c0d1,A-659280,25-11-2024,NaN,Enterprise,62,0.0,0.0,True,True,False,False,annual,False
3,S-f81687,A-e7a1e2,23-11-2024,13-12-2024,Enterprise,5,995.0,11940.0,False,False,False,True,monthly,True
4,S-cff5a2,A-ba6516,10-01-2024,NaN,Enterprise,27,5373.0,64476.0,False,False,False,False,monthly,True


In [392]:
subscriptions_df.shape

(5500, 14)

In [393]:
print(f"Null counts: \n{subscriptions_df.isnull().sum()}\n")
print(f"Duplicate counts: {subscriptions_df.duplicated().sum()}\n")

Null counts: 
subscription_id         0
account_id              0
start_date              0
end_date             4964
plan_tier              29
seats                   0
mrr_amount             29
arr_amount             21
is_trial                0
upgrade_flag            0
downgrade_flag          0
churn_flag              0
billing_frequency       0
auto_renew_flag         0
dtype: int64

Duplicate counts: 112



In [394]:
# insight: this is a very serious matter as end_date is a helpful column so for it to have (approx.) 90% null values, this can't be ignored
subscriptions_df[subscriptions_df.duplicated(subset=['account_id'])].sort_values(['account_id'])

,subscription_id,account_id,start_date,end_date,plan_tier,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag
3774,S-585d4c,A-00bed1,06-05-2024,NaN,Enterprise,28,5572.0,66864.0,False,True,False,False,monthly,True
3857,S-4a724b,A-00bed1,06-08-2024,NaN,Pro,28,1372.0,16464.0,False,False,False,False,annual,True
3126,S-289702,A-00bed1,01-09-2024,NaN,Enterprise,28,5572.0,66864.0,False,False,False,False,annual,False
678,S-a86344,A-00bed1,18-08-2024,NaN,Pro,36,1764.0,21168.0,False,True,False,False,annual,True
5231,S-585d4c,A-00bed1,06-05-2024,NaN,Enterprise,28,5572.0,66864.0,False,True,False,False,monthly,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3160,S-b192d9,A-ffdfd5,09-09-2024,NaN,Pro,14,0.0,0.0,True,True,False,False,monthly,True
2170,S-0a5cd7,A-ffdfd5,16-11-2024,NaN,Enterprise,30,5970.0,71640.0,False,False,False,False,annual,True
2139,S-bc5755,A-ffdfd5,20-10-2024,NaN,Enterprise,7,1393.0,16716.0,False,False,False,False,monthly,True
1683,S-9b5be4,A-ffdfd5,23-10-2024,NaN,Enterprise,4,796.0,9552.0,False,False,False,False,monthly,False


In [395]:
subscriptions_df.loc[
    subscriptions_df['churn_flag'] == False
    & subscriptions_df['end_date'].isna(),
    'end_date'
] = 'Ongoing'

In [396]:
subscriptions_df.isnull().sum()
# insight: end_date now only contains 54 null values which is (approx.) 1%. As its corresponding churned value is true, this is data inconsistency
# however, removing the whole row can be lost of valuable information which can be used later for model training
# hence, mapping those NaN values as 'Unknown' for now
subscriptions_df.loc[
    subscriptions_df['end_date'].isna(),
    'end_date'
] = 'Unknown'

subscriptions_df.isnull().sum()

subscription_id       0
account_id            0
start_date            0
end_date              0
plan_tier            29
seats                 0
mrr_amount           29
arr_amount           21
is_trial              0
upgrade_flag          0
downgrade_flag        0
churn_flag            0
billing_frequency     0
auto_renew_flag       0
dtype: int64

In [397]:
subscriptions_df.loc[
    subscriptions_df.duplicated(subset=['account_id', 'start_date'], keep=False)
    & subscriptions_df['plan_tier'].isna()
].head()

,subscription_id,account_id,start_date,end_date,plan_tier,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag
5051,S-000022,A-f03140,17-07-2024,Ongoing,NaN,27,5373.0,64476.0,False,False,True,False,monthly,True
5072,S-2573e4,A-7f8241,25-01-2024,Ongoing,NaN,60,1140.0,13680.0,False,False,False,False,monthly,True
5092,S-000038,A-54ecc2,30-10-2024,Ongoing,NaN,28,5572.0,66864.0,False,False,False,False,monthly,True
5100,S-00003e,A-e89151,23-05-2024,Ongoing,NaN,8,152.0,1824.0,False,True,False,False,monthly,True
5111,S-000047,A-de24d5,09-12-2024,21-12-2024,NaN,77,3773.0,45276.0,False,False,False,True,annual,True


In [398]:
subscriptions_df = subscriptions_df[
    ~(
        (subscriptions_df.duplicated(subset=['account_id', 'start_date'], keep=False))
        & (subscriptions_df['plan_tier'].isna())
    )
]

subscriptions_df = subscriptions_df[
    ~(
        (subscriptions_df.duplicated(subset=['account_id', 'start_date'], keep=False))
        & (subscriptions_df['mrr_amount'].isna())
    )
]

subscriptions_df = subscriptions_df[
    ~(
        (subscriptions_df.duplicated(subset=['account_id', 'start_date'], keep=False))
        & (subscriptions_df['arr_amount'].isna())
    )
]

subscriptions_df.isnull().sum()

subscription_id      0
account_id           0
start_date           0
end_date             0
plan_tier            0
seats                0
mrr_amount           0
arr_amount           0
is_trial             0
upgrade_flag         0
downgrade_flag       0
churn_flag           0
billing_frequency    0
auto_renew_flag      0
dtype: int64

In [399]:
subscriptions_df = subscriptions_df.drop_duplicates()
print(f"Duplicate counts: {subscriptions_df.duplicated().sum()}\n")

Duplicate counts: 0



In [400]:
subscriptions_df.info()

<class 'pandas.DataFrame'>
Index: 5309 entries, 0 to 5498
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   subscription_id    5309 non-null   str    
 1   account_id         5309 non-null   str    
 2   start_date         5309 non-null   str    
 3   end_date           5309 non-null   str    
 4   plan_tier          5309 non-null   str    
 5   seats              5309 non-null   int64  
 6   mrr_amount         5309 non-null   float64
 7   arr_amount         5309 non-null   float64
 8   is_trial           5309 non-null   bool   
 9   upgrade_flag       5309 non-null   bool   
 10  downgrade_flag     5309 non-null   bool   
 11  churn_flag         5309 non-null   bool   
 12  billing_frequency  5309 non-null   str    
 13  auto_renew_flag    5309 non-null   bool   
dtypes: bool(5), float64(2), int64(1), str(6)
memory usage: 440.7 KB


In [401]:
subscriptions_df.describe()

,seats,mrr_amount,arr_amount
count,5309.000000,5309.000000,5309.000000
mean,29.850819,2273.550198,27282.602373
std,23.072385,3424.220304,41090.643652
min,1.000000,0.000000,0.000000
25%,14.000000,285.000000,3420.000000
50%,24.000000,931.000000,11172.000000
75%,40.000000,2786.000000,33432.000000
max,189.000000,33830.000000,405960.000000


## *Support Tickets*

In [402]:
support_tickets_df.head()

,ticket_id,account_id,submitted_at,closed_at,resolution_time_hours,priority,first_response_time_minutes,satisfaction_score,escalation_flag
0,T-0024de,A-712f1c,27-07-2023,28-07-2023 03:00,27,high,74,NaN,False
1,T-4d04b9,A-e43bf7,08-07-2024,09-07-2024 03:00,27,urgent,144,NaN,False
2,T-d5e12f,A-0f3e88,17-10-2024,17-10-2024 19:00,19,urgent,93,4.0,False
3,T-dfce9a,A-4c56c9,08-09-2024,09-09-2024 23:00,47,medium,126,5.0,False
4,T-c59f77,A-6f8ad2,30-11-2024,01-12-2024 02:00,26,medium,8,NaN,False


In [403]:
support_tickets_df.shape

(2200, 9)

In [404]:
print(f"Null counts: \n{support_tickets_df.isnull().sum()}\n")
print(f"Duplicate counts: {support_tickets_df.duplicated().sum()}\n")

Null counts: 
ticket_id                        0
account_id                       0
submitted_at                     0
closed_at                        0
resolution_time_hours            0
priority                        30
first_response_time_minutes      0
satisfaction_score             931
escalation_flag                  0
dtype: int64

Duplicate counts: 33



In [405]:
support_tickets_df[support_tickets_df['priority'].isnull()].head()

,ticket_id,account_id,submitted_at,closed_at,resolution_time_hours,priority,first_response_time_minutes,satisfaction_score,escalation_flag
2001,T-000002,A-f25509,27-03-2024,27-03-2024 20:00,20,NaN,147,4.0,False
2009,T-d01651,A-7988d1,28-11-2023,30-11-2023 04:00,52,NaN,61,5.0,False
2017,T-00000e,A-c1dfe1,28-01-2024,29-01-2024 09:00,33,NaN,38,NaN,False
2023,T-000012,A-1b707d,18-10-2024,19-10-2024 08:00,32,NaN,160,NaN,False
2044,T-000023,A-186a44,30-12-2024,31-12-2024 00:00,24,NaN,146,NaN,False


In [406]:
support_tickets_df = support_tickets_df[
    ~(
        (support_tickets_df.duplicated(subset=['account_id', 'submitted_at'], keep=False)) 
        & (support_tickets_df['priority'].isna())
    )
]

support_tickets_df = support_tickets_df[
    ~(
        (support_tickets_df.duplicated(subset=['account_id', 'submitted_at'], keep=False)) 
        & (support_tickets_df['satisfaction_score'].isna())
    )
]

support_tickets_df.isnull().sum()

ticket_id                        0
account_id                       0
submitted_at                     0
closed_at                        0
resolution_time_hours            0
priority                         0
first_response_time_minutes      0
satisfaction_score             760
escalation_flag                  0
dtype: int64

In [407]:
support_tickets_df[support_tickets_df['satisfaction_score'].isnull()].head()

,ticket_id,account_id,submitted_at,closed_at,resolution_time_hours,priority,first_response_time_minutes,satisfaction_score,escalation_flag
0,T-0024de,A-712f1c,27-07-2023,28-07-2023 03:00,27,high,74,NaN,False
1,T-4d04b9,A-e43bf7,08-07-2024,09-07-2024 03:00,27,urgent,144,NaN,False
4,T-c59f77,A-6f8ad2,30-11-2024,01-12-2024 02:00,26,medium,8,NaN,False
5,T-90f06d,A-94c3cd,27-07-2023,27-07-2023 09:00,9,medium,60,NaN,False
8,T-7119c9,A-b179bf,27-08-2023,27-08-2023 16:00,16,urgent,154,NaN,False


In [408]:
support_tickets_df = support_tickets_df.drop_duplicates()
print(f"Null counts: \n{support_tickets_df.isnull().sum()}\n")
print(f"Duplicate counts: {support_tickets_df.duplicated().sum()}\n")

Null counts: 
ticket_id                        0
account_id                       0
submitted_at                     0
closed_at                        0
resolution_time_hours            0
priority                         0
first_response_time_minutes      0
satisfaction_score             760
escalation_flag                  0
dtype: int64

Duplicate counts: 0



In [409]:
support_tickets_df[support_tickets_df['closed_at'].isnull()].value_counts().sum()

np.int64(0)

In [410]:
# as satisfaction score is unknown and at large, mapping the nan values to -1
support_tickets_df.loc[
    support_tickets_df['satisfaction_score'].isna(),
    'satisfaction_score'
] = -1.0



In [411]:
print(f"Null counts: \n{support_tickets_df.isnull().sum()}\n")
print(f"Duplicate counts: {support_tickets_df.duplicated().sum()}\n")

Null counts: 
ticket_id                      0
account_id                     0
submitted_at                   0
closed_at                      0
resolution_time_hours          0
priority                       0
first_response_time_minutes    0
satisfaction_score             0
escalation_flag                0
dtype: int64

Duplicate counts: 0



In [412]:
support_tickets_df.info()

<class 'pandas.DataFrame'>
Index: 1996 entries, 0 to 2198
Data columns (total 9 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   ticket_id                    1996 non-null   str    
 1   account_id                   1996 non-null   str    
 2   submitted_at                 1996 non-null   str    
 3   closed_at                    1996 non-null   str    
 4   resolution_time_hours        1996 non-null   int64  
 5   priority                     1996 non-null   str    
 6   first_response_time_minutes  1996 non-null   int64  
 7   satisfaction_score           1996 non-null   float64
 8   escalation_flag              1996 non-null   bool   
dtypes: bool(1), float64(1), int64(2), str(5)
memory usage: 142.3 KB


In [413]:
support_tickets_df.describe()

,resolution_time_hours,first_response_time_minutes,satisfaction_score
count,1996.000000,1996.000000,1996.000000
mean,35.515030,88.498497,2.085671
std,21.862988,51.604244,2.502565
min,-71.000000,1.000000,-1.000000
25%,17.000000,43.000000,-1.000000
50%,35.000000,88.000000,3.000000
75%,54.000000,132.000000,4.000000
max,72.000000,180.000000,5.000000


In [419]:
# min value of resolution time hours is in negative which should not be possible
support_tickets_df[support_tickets_df['resolution_time_hours'] < 0]

,ticket_id,account_id,submitted_at,closed_at,resolution_time_hours,priority,first_response_time_minutes,satisfaction_score,escalation_flag
2005,T-4b2bf5,A-e7a1e2,31-03-2024,01-04-2024 21:00,-45,low,37,4.0,False
2015,T-7a48cd,A-4c38bc,27-07-2024,29-07-2024 18:00,-66,medium,37,4.0,False
2061,T-000030,A-781cc0,28-01-2024,30-01-2024 23:00,-71,low,9,3.0,False
2111,T-000050,A-55f257,12-07-2024,14-07-2024 17:00,-65,low,13,5.0,False
2129,T-00005a,A-4a267b,17-05-2023,17-05-2023 07:00,-7,low,166,5.0,False
2131,T-00005c,A-32fb14,05-09-2024,05-09-2024 07:00,-7,low,89,4.0,False
2135,T-2fc301,A-7c4956,10-06-2024,10-06-2024 11:00,-11,high,102,4.0,False
2137,T-b3f296,A-05f0e5,04-10-2023,04-10-2023 11:00,-11,high,91,4.0,False
2147,T-000067,A-4a4c2d,15-11-2024,16-11-2024 20:00,-44,high,128,5.0,False
2154,T-00006b,A-fb186e,11-01-2023,12-01-2023 19:00,-43,medium,132,3.0,False


In [420]:
support_tickets_df['resolution_time_hours'] = support_tickets_df['resolution_time_hours'].abs()

support_tickets_df.describe()

,resolution_time_hours,first_response_time_minutes,satisfaction_score
count,1996.000000,1996.000000,1996.000000
mean,35.968938,88.498497,2.085671
std,21.107507,51.604244,2.502565
min,1.000000,1.000000,-1.000000
25%,17.000000,43.000000,-1.000000
50%,35.000000,88.000000,3.000000
75%,55.000000,132.000000,4.000000
max,72.000000,180.000000,5.000000


# Export Clean Datasets

In [424]:
dfs_to_save = [accounts_df, churn_events_df, feature_usage_df, subscriptions_df, support_tickets_df]
cleaned_name = [
    'accounts_cleaned.csv', 
    'churn_events_cleaned.csv', 
    'feature_usage_cleaned.csv', 
    'subscriptions_cleaned.csv', 
    'support_tickets_cleaned.csv']

save_path = f"{gl_path}\\data\\cleaned"

for i, d in enumerate(dfs_to_save):
    d.to_csv(f"{save_path}\\{cleaned_name[i]}")